# 52. The EOQ with Quantity Discounts Problem

## Tier 3: The Advanced Algorithm (Metaheuristic Implementation)

### Key assumptions
- Annual demand is constant and known
- Ordering cost per order is constant
- Holding cost is a percentage of unit cost
- Quantity discounts are piecewise constant
- Additional constraints like storage capacity limits may exist
- Complex multi-tier discount structures with potential non-linear relationships
- Metaheuristic can handle complex constraints and objective functions

### Approach (step-by-step)
For complex multi-tier discount structures with additional constraints such as storage capacity limits, supplier reliability factors, or seasonal demand variations, we employ the Water Cycle Algorithm (WCA), a nature-inspired metaheuristic that mimics the natural water cycle process of evaporation, condensation, and precipitation.

**Water Cycle Algorithm Theory:**

The Water Cycle Algorithm models the optimization process as water molecules moving through the natural hydrological cycle. Solutions (raindrops) flow toward better solutions (rivers and seas) through gravitational attraction, with evaporation and precipitation providing exploration and exploitation mechanisms.

Key components:
- **Sea**: The best solution found so far
- **Rivers**: Good quality solutions that attract nearby raindrops
- **Raindrops**: Current population of candidate solutions
- **Evaporation**: Mechanism for escaping local optima
- **Raining**: Random generation of new solutions

### What to look for in the results
- Ability to handle complex constraints (storage limits, etc.)
- Discovery of non-obvious optimal solutions in complex search spaces
- Convergence behavior and solution quality over iterations
- Performance comparison with simpler approaches
- Robustness to problem complexity and constraints

### Concrete example (from the source)
Complex discount structure with storage constraints:
- 5 discount tiers: (0-499): $12.00, (500-999): $10.50, (1000-1999): $9.25, (2000-4999): $8.75, (5000+): $8.00
- Storage capacity limit: 3,000 units
- Demand: 5,000 units/year, S = $75, h = 25%

### Visualization(s)
We will create visualizations showing:
- Water Cycle Algorithm convergence process
- Population dynamics (sea, rivers, raindrops)
- Solution quality improvement over iterations
- Comparison with classical approaches
- Constraint handling visualization

### Why this Tier exists vs earlier Tiers
This tier addresses the limitations of both mathematical optimization and heuristic approaches when dealing with complex constraints, non-linear relationships, and large search spaces. While Tier 1 provides exact solutions for simple problems and Tier 2 offers efficient heuristics for standard cases, this metaheuristic approach can handle complex real-world scenarios with multiple constraints and non-obvious solution spaces.

### Pros / Cons vs Tier 1 & 2
**Pros:**
- Handles complex constraints (storage limits, reliability factors, etc.)
- Suitable for non-linear and discontinuous objective functions
- Robust to local optima through evaporation mechanism
- Scalable to very large problem instances
- Can find creative solutions that simpler methods miss
- Parallelizable for improved performance

**Cons:**
- No guarantee of global optimality
- Requires parameter tuning (population size, iterations, etc.)
- Computationally more intensive than simple heuristics
- Solution quality may vary between runs
- More complex to implement and understand
- May require multiple runs for consistent results

### When to use this Tier
- When storage capacity constraints are binding
- For complex multi-tier discount structures (5+ tiers)
- When additional constraints exist (reliability, seasonal factors)
- For large-scale inventory optimization problems
- When traditional methods fail to find feasible solutions
- In research and development of advanced inventory systems

In [1]:
from typing import Tuple, List, Dict, Set
from itertools import product

# Import required libraries
import numpy as np
import random
import math
from typing import List, Tuple
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from dataclasses import dataclass
import time
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
@dataclass
class DiscountTier:
    """Represents a discount tier with quantity range and unit cost"""
    min_qty: int
    max_qty: float
    unit_cost: float
    tier_id: int
    
    def contains_quantity(self, quantity: float) -> bool:
        return self.min_qty <= quantity <= self.max_qty

class WaterCycleEOQ:
    """Water Cycle Algorithm for EOQ with quantity discounts and constraints"""
    
    def __init__(self, demand, ordering_cost, holding_rate, discount_tiers,
                 storage_capacity=float('inf')):
        self.demand = demand
        self.ordering_cost = ordering_cost
        self.holding_rate = holding_rate
        self.discount_tiers = discount_tiers
        self.storage_capacity = storage_capacity
        
        # WCA parameters
        self.population_size = 50
        self.max_iterations = 100
        self.num_rivers = 4
        self.evaporation_condition = 1e-6
        
        # Track convergence
        self.convergence_history = []
        
    def get_unit_cost(self, quantity):
        """Determine unit cost based on quantity and discount structure"""
        for min_qty, max_qty, unit_cost in self.discount_tiers:
            if min_qty <= quantity <= max_qty:
                return unit_cost
        return self.discount_tiers[-1][2]  # Default to highest tier
    
    def evaluate_solution(self, quantity):
        """Calculate total annual cost with constraints"""
        # Check constraints
        if quantity <= 0 or quantity > self.storage_capacity:
            return float('inf')  # Infeasible solution
        
        unit_cost = self.get_unit_cost(quantity)
        holding_cost = (quantity * unit_cost * self.holding_rate) / 2
        ordering_cost = (self.demand * self.ordering_cost) / quantity
        purchase_cost = self.demand * unit_cost
        
        return holding_cost + ordering_cost + purchase_cost
    
    def initialize_population(self):
        """Initialize population of raindrops, rivers, and sea"""
        # Generate random solutions
        population = []
        for _ in range(self.population_size):
            # Random quantity between reasonable bounds
            min_bound = 1
            max_bound = min(self.storage_capacity, self.demand)
            quantity = random.uniform(min_bound, max_bound)
            cost = self.evaluate_solution(quantity)
            population.append([quantity, cost])
        
        # Sort by cost (ascending - best solutions first)
        population.sort(key=lambda x: x[1])
        
        # Assign sea (best solution)
        sea = population[0]
        
        # Assign rivers (next best solutions)
        rivers = population[1:self.num_rivers + 1] if len(population) > 1 else []
        
        # Assign raindrops (remaining solutions)
        raindrops = population[self.num_rivers + 1:] if len(population) > self.num_rivers + 1 else []
        
        return sea, rivers, raindrops
    
    def calculate_flow_intensity(self, solutions):
        """Calculate flow intensity for each river and sea"""
        if not solutions:
            return []
        
        costs = [sol[1] for sol in solutions]
        max_cost = max(costs)
        # Invert costs so better solutions have higher flow
        flows = [max_cost - cost + 1 for cost in costs]
        total_flow = sum(flows)
        return [flow / total_flow for flow in flows]
    
    def flow_raindrops_to_rivers(self, raindrops, rivers, sea):
        """Move raindrops toward rivers and sea based on flow intensity"""
        if not raindrops:
            return []
        
        all_attractors = [sea] + rivers
        flows = self.calculate_flow_intensity(all_attractors)
        
        # Assign raindrops to attractors based on flow intensity
        attractor_assignments = []
        for flow in flows:
            num_assigned = max(1, int(flow * len(raindrops)))
            attractor_assignments.append(num_assigned)
        
        # Adjust to match total raindrops
        while sum(attractor_assignments) < len(raindrops):
            for i in range(len(attractor_assignments)):
                if sum(attractor_assignments) < len(raindrops):
                    attractor_assignments[i] += 1
                else:
                    break
        
        while sum(attractor_assignments) > len(raindrops):
            max_idx = attractor_assignments.index(max(attractor_assignments))
            attractor_assignments[max_idx] -= 1
        
        # Move raindrops toward their assigned attractors
        new_raindrops = []
        idx = 0
        for i, num_assigned in enumerate(attractor_assignments):
            attractor = all_attractors[i]
            for j in range(num_assigned):
                if idx < len(raindrops):
                    raindrop = raindrops[idx]
                    # Move toward attractor with some randomness
                    c = 2  # Acceleration coefficient
                    rand = random.uniform(0, 1)
                    new_quantity = raindrop + c * rand * (attractor - raindrop)
                    
                    # Ensure bounds
                    new_quantity = max(1, min(self.storage_capacity, new_quantity))
                    new_cost = self.evaluate_solution(new_quantity)
                    
                    new_raindrops.append([new_quantity, new_cost])
                    idx += 1
        
        return new_raindrops
    
    def flow_rivers_to_sea(self, rivers, sea):
        """Move rivers toward sea"""
        new_rivers = []
        for river in rivers:
            c = 2
            rand = random.uniform(0, 1)
            new_quantity = river + c * rand * (sea - river)
            new_quantity = max(1, min(self.storage_capacity, new_quantity))
            new_cost = self.evaluate_solution(new_quantity)
            
            # River becomes sea if better
            if new_cost < sea[1]:
                new_rivers.append(sea[:])
                sea = [new_quantity, new_cost]
            else:
                new_rivers.append([new_quantity, new_cost])
        
        return new_rivers, sea
    
    def evaporation_condition_check(self, river, sea):
        """Check if river should evaporate (too close to sea)"""
        return abs(river - sea) < self.evaporation_condition
    
    def raining_process(self, num_new_raindrops):
        """Generate new raindrops through raining"""
        new_raindrops = []
        for _ in range(num_new_raindrops):
            quantity = random.uniform(1, min(self.storage_capacity, self.demand))
            cost = self.evaluate_solution(quantity)
            new_raindrops.append([quantity, cost])
        return new_raindrops
    
    def optimize(self, track_convergence=True):
        """Main Water Cycle Algorithm optimization loop"""
        # Initialize population
        sea, rivers, raindrops = self.initialize_population()
        best_solution = sea[:]
        
        if track_convergence:
            self.convergence_history = [best_solution[1]]
        
        for iteration in range(self.max_iterations):
            # Flow raindrops to rivers and sea
            raindrops = self.flow_raindrops_to_rivers(raindrops, rivers, sea)
            
            # Flow rivers to sea
            rivers, sea = self.flow_rivers_to_sea(rivers, sea)
            
            # Update best solution
            if sea[1] < best_solution[1]:
                best_solution = sea[:]
            
            # Evaporation and raining process
            evaporated_rivers = []
            remaining_rivers = []
            
            for river in rivers:
                if self.evaporation_condition_check(river, sea):
                    evaporated_rivers.append(river)
                else:
                    remaining_rivers.append(river)
            
            # Generate new raindrops to replace evaporated rivers
            if evaporated_rivers:
                new_raindrops = self.raining_process(len(evaporated_rivers))
                raindrops.extend(new_raindrops)
            
            rivers = remaining_rivers
            
            # Add new rivers if needed
            if len(rivers) < self.num_rivers and raindrops:
                all_solutions = raindrops[:]
                all_solutions.sort(key=lambda x: x[1])
                needed_rivers = min(self.num_rivers - len(rivers), len(all_solutions))
                new_rivers = all_solutions[:needed_rivers]
                rivers.extend(new_rivers)
                raindrops = all_solutions[needed_rivers:]
            
            # Track convergence
            if track_convergence:
                self.convergence_history.append(best_solution[1])
        
        return best_solution, best_solution[1]

# Create the complex example from the source
print("=" * 80)
print("COMPLEX EOQ PROBLEM WITH STORAGE CONSTRAINTS")
print("=" * 80)

# Complex discount structure with storage constraints (from source)
demand = 5000
ordering_cost = 75
holding_rate = 0.25
storage_capacity = 3000  # Storage constraint

discount_tiers = [
    (0, 499, 12.00),      # Tier 1
    (500, 999, 10.50),    # Tier 2
    (1000, 1999, 9.25),   # Tier 3
    (2000, 4999, 8.75),   # Tier 4
    (5000, float('inf'), 8.00)  # Tier 5
]

print(f"\nComplex Problem Parameters:")
print(f"Annual Demand: {demand} units/year")
print(f"Ordering Cost: ${ordering_cost}/order")
print(f"Holding Rate: {holding_rate*100:.0f}% of unit cost")
print(f"Storage Capacity: {storage_capacity} units")
print("\nDiscount Tiers:")
for i, (min_qty, max_qty, unit_cost) in enumerate(discount_tiers):
    max_str = f"{max_qty:.0f}" if max_qty != float('inf') else "∞"
    print(f"Tier {i+1}: {min_qty}-{max_str} units at ${unit_cost:.2f} each")

COMPLEX EOQ PROBLEM WITH STORAGE CONSTRAINTS

Complex Problem Parameters:
Annual Demand: 5000 units/year
Ordering Cost: $75/order
Holding Rate: 25% of unit cost
Storage Capacity: 3000 units

Discount Tiers:
Tier 1: 0-499 units at $12.00 each
Tier 2: 500-999 units at $10.50 each
Tier 3: 1000-1999 units at $9.25 each
Tier 4: 2000-4999 units at $8.75 each
Tier 5: 5000-∞ units at $8.00 each


In [ ]:
try:
    # Solve using Water Cycle Algorithm
    wca_solver = WaterCycleEOQ(demand, ordering_cost, holding_rate, 
                               discount_tiers, storage_capacity)
    
    print("\n" + "=" * 50)
    print("WATER CYCLE ALGORITHM OPTIMIZATION")
    print("=" * 50)
    
    # Run optimization
    start_time = time.time()
    optimal_quantity, optimal_cost = wca_solver.optimize()
    computation_time = time.time() - start_time
    
    print(f"\nWater Cycle Algorithm Results:")
    print(f"Optimal Order Quantity: {optimal_quantity:.0f} units")
    print(f"Minimum Total Annual Cost: ${optimal_cost:.2f}")
    print(f"Unit Cost: ${wca_solver.get_unit_cost(optimal_quantity):.2f}")
    print(f"Computation Time: {computation_time:.3f} seconds")
    
    # Check if storage constraint is binding
    is_storage_binding = optimal_quantity >= storage_capacity * 0.95
    print(f"Storage Constraint Binding: {'Yes' if is_storage_binding else 'No'}")
    
    # Determine which tier the solution falls in
    selected_tier = None
    for i, (min_qty, max_qty, unit_cost) in enumerate(discount_tiers):
        if min_qty <= optimal_quantity <= max_qty:
            selected_tier = i + 1
            break
    
    print(f"Selected Discount Tier: {selected_tier}")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: unsupported operand type(s) for -: 'list' and 'list'...]

In [ ]:
try:
    # Compare with classical approach for validation
    def classical_eoq_approach(demand, ordering_cost, holding_rate, discount_tiers, storage_capacity=float('inf')):
        """Classical three-step heuristic approach for comparison"""
        results = []
        
        for min_qty, max_qty, unit_cost in discount_tiers:
            # Calculate EOQ
            eoq = math.sqrt((2 * demand * ordering_cost) / (holding_rate * unit_cost))
            
            # Adjust for constraints
            if eoq < min_qty:
                adjusted_qty = min_qty
            elif max_qty != float('inf') and eoq > max_qty:
                adjusted_qty = max_qty
            else:
                adjusted_qty = eoq
            
            # Apply storage constraint
            if adjusted_qty > storage_capacity:
                adjusted_qty = storage_capacity
            
            # Calculate total cost
            if adjusted_qty > 0:
                unit_cost_actual = unit_cost
                # Check if adjusted quantity still qualifies for the discount
                if not (min_qty <= adjusted_qty <= max_qty):
                    # Find appropriate tier for adjusted quantity
                    for min_q, max_q, unit_c in discount_tiers:
                        if min_q <= adjusted_qty <= max_q:
                            unit_cost_actual = unit_c
                            break
                
                holding_cost = (adjusted_qty * unit_cost_actual * holding_rate) / 2
                ordering_cost = (demand * ordering_cost) / adjusted_qty
                purchase_cost = demand * unit_cost_actual
                total_cost = holding_cost + ordering_cost + purchase_cost
                
                results.append({
                    'adjusted_qty': adjusted_qty,
                    'total_cost': total_cost,
                    'unit_cost': unit_cost_actual,
                    'tier': discount_tiers.index((min_qty, max_qty, unit_cost)) + 1
                })
        
        # Find best solution
        if results:
            best = min(results, key=lambda x: x['total_cost'])
            return best['adjusted_qty'], best['total_cost'], best['tier'], best['unit_cost']
        else:
            return None, None, None, None
    
    # Classical approach
    classical_qty, classical_cost, classical_tier, classical_unit_cost = classical_eoq_approach(
        demand, ordering_cost, holding_rate, discount_tiers, storage_capacity
    )
    
    print(f"\nComparison with Classical Approach:")
    print("-" * 50)
    print(f"Classical Approach:")
    print(f"  Order Quantity: {classical_qty:.0f} units")
    print(f"  Total Cost: ${classical_cost:.2f}")
    print(f"  Tier: {classical_tier}")
    
    print(f"\nWater Cycle Algorithm:")
    print(f"  Order Quantity: {optimal_quantity:.0f} units")
    print(f"  Total Cost: ${optimal_cost:.2f}")
    print(f"  Tier: {selected_tier}")
    
    # Calculate improvement
    if classical_cost and optimal_cost < classical_cost:
        improvement = classical_cost - optimal_cost
        improvement_pct = (improvement / classical_cost) * 100
        print(f"\nWCA Improvement: ${improvement:.2f} ({improvement_pct:.2f}%)")
    else:
        print(f"\nWCA matches or improves classical solution")
    
    # Detailed tier-by-tier comparison (from source)
    print(f"\nExpected Results from Source:")
    print(f"Tier 1: Q=500, Cost=$65,610.00")
    print(f"Tier 2: Q=690, Cost=$58,144.33")
    print(f"Tier 3: Q=1309, Cost=$50,935.48")
    print(f"Tier 4: Q=2000, Cost=$48,656.25")
    print(f"WCA Expected: Q=1837, Cost=$48,426.35")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'optimal_quantity' is not defined...]

In [ ]:
try:
    # Create comprehensive visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Water Cycle Algorithm for EOQ Optimization', fontsize=16, fontweight='bold')
    
    # 1. Convergence history
    iterations = range(len(wca_solver.convergence_history))
    ax1.plot(iterations, wca_solver.convergence_history, 'b-', linewidth=2, alpha=0.7)
    ax1.scatter([0, len(iterations)-1], [wca_solver.convergence_history[0], wca_solver.convergence_history[-1]], 
               color='red', s=100, zorder=5)
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Best Total Cost ($)')
    ax1.set_title('WCA Convergence History')
    ax1.grid(True, alpha=0.3)
    ax1.annotate(f'Start: ${wca_solver.convergence_history[0]:.0f}', 
                 xy=(0, wca_solver.convergence_history[0]),
                 xytext=(10, wca_solver.convergence_history[0]*1.02),
                 fontsize=9, color='red')
    ax1.annotate(f'Best: ${wca_solver.convergence_history[-1]:.0f}', 
                 xy=(len(iterations)-1, wca_solver.convergence_history[-1]),
                 xytext=(len(iterations)-30, wca_solver.convergence_history[-1]*1.02),
                 fontsize=9, color='red')
    
    # 2. Cost landscape with optimal solution
    quantities = np.linspace(100, storage_capacity, 200)
    costs = []
    for q in quantities:
        cost = wca_solver.evaluate_solution(q)
        costs.append(cost if cost != float('inf') else np.nan)
    
    ax2.plot(quantities, costs, 'lightblue', linewidth=2, alpha=0.7, label='Cost Landscape')
    
    # Mark discount tier boundaries
    for i, (min_qty, max_qty, unit_cost) in enumerate(discount_tiers):
        if max_qty <= storage_capacity:
            ax2.axvspan(min_qty, max_qty, alpha=0.1, color=f'C{i}', label=f'Tier {i+1}' if i < 3 else '')
    
    # Mark optimal solutions
    ax2.plot(optimal_quantity, optimal_cost, 'r*', markersize=20, label=f'WCA Optimal: {optimal_quantity:.0f}')
    if classical_qty:
        ax2.plot(classical_qty, classical_cost, 'g^', markersize=15, label=f'Classical: {classical_qty:.0f}')
    
    # Mark storage constraint
    ax2.axvline(storage_capacity, color='red', linestyle='--', alpha=0.7, label=f'Storage Limit: {storage_capacity}')
    
    ax2.set_xlabel('Order Quantity (units)')
    ax2.set_ylabel('Total Annual Cost ($)')
    ax2.set_title('Cost Landscape with Constraints')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Population dynamics visualization
    def visualize_population_dynamics():
        """Create a snapshot of population dynamics"""
        sea, rivers, raindrops = wca_solver.initialize_population()
        
        # Create positions for visualization
        positions = []
        labels = []
        colors = []
        
        # Sea (best solution)
        positions.append(sea[0])
        labels.append(f'Sea\n{sea[0]:.0f}\n${sea[1]:.0f}')
        colors.append('darkblue')
        
        # Rivers
        for i, river in enumerate(rivers[:4]):  # Show up to 4 rivers
            positions.append(river[0])
            labels.append(f'R{i+1}\n{river[0]:.0f}\n${river[1]:.0f}')
            colors.append('blue')
        
        # Sample of raindrops
        sample_raindrops = random.sample(raindrops, min(10, len(raindrops))) if raindrops else []
        for drop in sample_raindrops:
            positions.append(drop[0])
            labels.append(f'{drop[0]:.0f}')
            colors.append('lightblue')
        
        return positions, labels, colors
    
    positions, labels, colors = visualize_population_dynamics()
    y_pos = np.random.uniform(0.5, 1, len(positions))  # Random y for visualization
    
    ax3.scatter(positions, y_pos, c=colors, s=100, alpha=0.7)
    for i, (x, y, label) in enumerate(zip(positions, y_pos, labels)):
        if i < 5:  # Only label sea and rivers
            ax3.annotate(label, (x, y), xytext=(x, y + 0.05), 
                        ha='center', fontsize=8, fontweight='bold' if i == 0 else 'normal')
    
    ax3.set_xlabel('Order Quantity (units)')
    ax3.set_ylabel('Population Hierarchy')
    ax3.set_title('Water Cycle Population Dynamics')
    ax3.set_ylim(0.4, 1.1)
    ax3.grid(True, alpha=0.3)
    
    # 4. Performance comparison
    # Test with different problem complexities
    complexities = ['Simple (2 tiers)', 'Medium (3 tiers)', 'Complex (5 tiers)', 'Very Complex (8 tiers)']
    wca_times = []
    classical_times = []
    wca_costs = []
    classical_costs = []
    
    for complexity in complexities:
        if 'Simple' in complexity:
            tiers = discount_tiers[:2]
        elif 'Medium' in complexity:
            tiers = discount_tiers[:3]
        elif 'Complex' in complexity:
            tiers = discount_tiers
        else:  # Very Complex
            tiers = discount_tiers + [
                (5000, 9999, 7.50),
                (10000, float('inf'), 7.00)
            ]
        
        # Time WCA
        start = time.time()
        wca_test = WaterCycleEOQ(demand, ordering_cost, holding_rate, tiers, storage_capacity)
        wca_test.max_iterations = 50  # Reduce for speed
        _, wca_cost = wca_test.optimize()
        wca_times.append(time.time() - start)
        wca_costs.append(wca_cost)
        
        # Time Classical
        start = time.time()
        _, classical_cost, _, _ = classical_eoq_approach(demand, ordering_cost, holding_rate, tiers, storage_capacity)
        classical_times.append(time.time() - start)
        classical_costs.append(classical_cost)
    
    x_pos = np.arange(len(complexities))
    width = 0.35
    
    ax4_twin = ax4.twinx()
    bars1 = ax4.bar(x_pos - width/2, wca_times, width, label='WCA Time', alpha=0.8, color='skyblue')
    bars2 = ax4.bar(x_pos + width/2, classical_times, width, label='Classical Time', alpha=0.8, color='lightcoral')
    
    # Add cost lines
    ax4_twin.plot(x_pos, wca_costs, 'o-', color='blue', linewidth=2, markersize=6, label='WCA Cost')
    ax4_twin.plot(x_pos, classical_costs, 's-', color='red', linewidth=2, markersize=6, label='Classical Cost')
    
    ax4.set_xlabel('Problem Complexity')
    ax4.set_ylabel('Computation Time (seconds)')
    ax4_twin.set_ylabel('Total Cost ($)')
    ax4.set_title('Performance vs Complexity')
    ax4.set_xticks(x_pos)
    ax4.set_xticklabels([c.split('(')[0] for c in complexities], rotation=45)
    ax4.legend(loc='upper left')
    ax4_twin.legend(loc='upper right')
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'optimal_quantity' is not defined...]

In [ ]:
try:
    # Advanced analysis: Constraint handling and robustness
    print("=" * 80)
    print("ADVANCED ANALYSIS: CONSTRAINT HANDLING & ROBUSTNESS")
    print("=" * 80)
    
    # Test different storage capacity scenarios
    storage_scenarios = [1000, 2000, 3000, 4000, 5000, float('inf')]
    scenario_results = []
    
    for storage in storage_scenarios:
        storage_str = f"{storage:.0f}" if storage != float('inf') else "Unlimited"
        
        # WCA solution
        wca_test = WaterCycleEOQ(demand, ordering_cost, holding_rate, discount_tiers, storage)
        wca_test.max_iterations = 50  # Reduce for speed
        wca_qty, wca_cost = wca_test.optimize()
        
        # Classical solution
        classical_qty, classical_cost, classical_tier, _ = classical_eoq_approach(
            demand, ordering_cost, holding_rate, discount_tiers, storage
        )
        
        scenario_results.append({
            'storage': storage_str,
            'wca_qty': wca_qty,
            'wca_cost': wca_cost,
            'classical_qty': classical_qty,
            'classical_cost': classical_cost,
            'improvement': classical_cost - wca_cost if classical_cost else 0,
            'improvement_pct': ((classical_cost - wca_cost) / classical_cost * 100) if classical_cost else 0
        })
    
    print("\nStorage Capacity Impact Analysis:")
    print("-" * 70)
    for result in scenario_results:
        print(f"Storage: {result['storage']:>8} | WCA: {result['wca_qty']:6.0f} @ ${result['wca_cost']:8.0f} | Classical: {result['classical_qty']:6.0f} @ ${result['classical_cost']:8.0f}")
        if result['improvement'] > 0:
            print(f"{'':>22} | Improvement: ${result['improvement']:.2f} ({result['improvement_pct']:.2f}%)")
        print()
    
    # Multiple runs for robustness testing
    print("Robustness Testing (Multiple Runs):")
    print("-" * 40)
    
    num_runs = 10
    wca_results = []
    for run in range(num_runs):
        wca_test = WaterCycleEOQ(demand, ordering_cost, holding_rate, discount_tiers, storage_capacity)
        qty, cost = wca_test.optimize()
        wca_results.append({'qty': qty, 'cost': cost})
    
    quantities = [r['qty'] for r in wca_results]
    costs = [r['cost'] for r in wca_results]
    
    print(f"WCA Results over {num_runs} runs:")
    print(f"Quantity - Mean: {np.mean(quantities):.0f}, Std: {np.std(quantities):.0f}")
    print(f"Cost - Mean: ${np.mean(costs):.2f}, Std: ${np.std(costs):.2f}")
    print(f"Coefficient of Variation: {np.std(costs)/np.mean(costs)*100:.2f}%")
    
    # Best and worst runs
    best_run = min(wca_results, key=lambda x: x['cost'])
    worst_run = max(wca_results, key=lambda x: x['cost'])
    print(f"Best run: {best_run['qty']:.0f} units @ ${best_run['cost']:.2f}")
    print(f"Worst run: {worst_run['qty']:.0f} units @ ${worst_run['cost']:.2f}")
    print(f"Range: ${worst_run['cost'] - best_run['cost']:.2f}")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: unsupported operand type(s) for -: 'list' and 'list'...]

In [ ]:
try:
    # Summary and Key Insights
    print("=" * 80)
    print("WATER CYCLE ALGORITHM SUMMARY")
    print("=" * 80)
    
    print(f"\nComplex Problem Solution:")
    print(f"- Optimal Order Quantity: {optimal_quantity:.0f} units")
    print(f"- Total Annual Cost: ${optimal_cost:,.2f}")
    print(f"- Selected Discount Tier: {selected_tier}")
    print(f"- Unit Cost: ${wca_solver.get_unit_cost(optimal_quantity):.2f}")
    print(f"- Storage Constraint: {storage_capacity} units")
    print(f"- Computation Time: {computation_time:.3f} seconds")
    
    if classical_cost:
        improvement = classical_cost - optimal_cost
        improvement_pct = (improvement / classical_cost) * 100 if improvement > 0 else 0
        print(f"\nPerformance vs Classical Approach:")
        print(f"- Cost Improvement: ${improvement:.2f} ({improvement_pct:.2f}%)")
        print(f"- Solution Quality: {'Better' if improvement > 0 else 'Equal'}")
    
    print(f"\nWater Cycle Algorithm Characteristics:")
    print(f"1. Population-based metaheuristic with natural inspiration")
    print(f"2. Handles complex constraints (storage limits, etc.)")
    print(f"3. Balances exploration (evaporation) and exploitation (flow)")
    print(f"4. Suitable for non-linear and discontinuous problems")
    print(f"5. Robust to local optima through population diversity")
    
    print(f"\nKey Advantages for EOQ Problems:")
    print(f"✓ Handles binding storage constraints effectively")
    print(f"✓ Discovers non-obvious optimal solutions")
    print(f"✓ Scales to complex multi-tier discount structures")
    print(f"✓ Robust to problem complexity and constraints")
    print(f"✓ Parallelizable for improved performance")
    
    print(f"\nPractical Applications:")
    print(f"• Large-scale inventory optimization with storage limits")
    print(f"• Multi-product coordination problems")
    print(f"• Supply chain design with capacity constraints")
    print(f"• Strategic procurement with complex discount structures")
    print(f"• Research and development of advanced inventory systems")
    
    print(f"\nAlgorithm Insights:")
    print(f"• Convergence typically achieved within 50-100 iterations")
    print(f"• Population diversity prevents premature convergence")
    print(f"• Evaporation mechanism enables escape from local optima")
    print(f"• Flow dynamics guide search toward promising regions")
    print(f"• Solution quality improves with problem complexity")
    
    print(f"\nWhen to Use Water Cycle Algorithm:")
    print(f"• Storage capacity constraints are binding")
    print(f"• Complex discount structures (5+ tiers)")
    print(f"• Non-linear cost functions or constraints")
    print(f"• Traditional methods fail to find feasible solutions")
    print(f"• Research and advanced inventory optimization")
    
    print(f"\nExpected Results from Source:")
    print(f"- WCA Optimal: Q=1837, Cost=$48,426.35")
    print(f"\nActual Results:")
    print(f"- WCA Optimal: Q={optimal_quantity:.0f}, Cost=${optimal_cost:.2f}")
    print(f"\nVerification: {'✓ Close to expected' if abs(optimal_quantity - 1837) < 100 else '✗ Needs adjustment'}")
except Exception as e:
    print(f'Error in cell: {e}')

[Error handled: name 'optimal_quantity' is not defined...]